# SAM3 Training Documentation

> Comprehensive guide to SAM3 (Segment Anything Model 3) training scripts, parameters, and implementation details

## 1. What is SAM3?

**SAM3 (Segment Anything Model 3)** is Meta AI's latest foundation model for promptable segmentation, released on November 19, 2025. It represents a major evolution in the Segment Anything series.

### Key Features

- **Open-vocabulary instance detection and segmentation**: Unlike SAM 1 and 2, SAM3 can detect and segment ALL instances of a concept specified by text prompts, image exemplars, or both
- **Unified model**: Works on both images and videos for detection, segmentation, and tracking
- **Text-based prompting**: You can simply tell SAM3 what to segment (e.g., "red baseball cap") and it will identify all matching objects
- **Model size**: 848M parameters
- **Performance**: 2x improvement over existing systems (Gemini 2.5 Pro, OWLv2) on the SA-Co benchmark

### Architecture Overview

SAM3 consists of two main components:

1. **Detector**: A DETR-based model conditioned on text, geometry, and image exemplars
2. **Tracker**: Uses the SAM 2 transformer encoder-decoder architecture
3. **Shared Vision Encoder**: Both components share a common vision encoder

### Training Dataset

SAM3 was trained on the **SA-Co (Segment Anything with Concepts)** dataset:
- 4 million automatically annotated concepts
- 270K unique concepts
- Available on HuggingFace and Roboflow

## 2. Installation and Setup

### Repository

SAM3 is available on GitHub: [facebookresearch/sam3](https://github.com/facebookresearch/sam3)

### Installation

```bash
# Clone the repository
git clone https://github.com/facebookresearch/sam3.git
cd sam3

# Install for training and development
pip install -e ".[train,dev]"
```

### Model Checkpoints

Model checkpoints are available on HuggingFace:
- HuggingFace: `facebook/sam3`
- Requires requesting access from Meta

### License

SAM3 is released under an open-source license.

## 3. Training Scripts and Commands

### Main Training Script

The primary training entry point is `sam3/train/train.py`.

### Basic Training Command

```bash
python sam3/train/train.py -c CONFIG_NAME [optional arguments]
```

### Training Configurations

#### Single GPU Training
```bash
python sam3/train/train.py -c your_config --use-cluster 0 --num-gpus 1
```

#### Multi-GPU Local Training
```bash
python sam3/train/train.py -c your_config --use-cluster 0 --num-gpus 4
```

#### Cluster Training (SLURM)
```bash
python sam3/train/train.py -c your_config --use-cluster 1
```

### Evaluation Mode

To run zero-shot evaluation on benchmark datasets:
```bash
python sam3/train/train.py -c your_config trainer.mode=val
```

## 4. Training Parameters and Hyperparameters

### Optimizer Configuration

SAM3 uses **AdamW optimizer** with component-specific learning rates:

#### Learning Rates (with lr_scale factor of 0.1)

| Component | Base LR | Scaled LR (actual) |
|-----------|---------|--------------------|
| Transformer | 8e-4 | 8e-5 |
| Vision Backbone | 2.5e-4 | 2.5e-5 |
| Language Backbone | 5e-5 | 5e-6 |

#### Optimizer Settings

```yaml
optimizer: AdamW
weight_decay: 0.1
gradient_clipping:
  max_norm: 0.1
  norm_type: 2
```

### Learning Rate Scheduler

SAM3 uses **InverseSquareRootParamScheduler**:

```yaml
scheduler_warmup_steps: 20
scheduler_timescale: 20
scheduler_cooldown_steps: 20
```

This scheduler provides:
- Linear warmup for the first 20 steps
- Inverse square root decay
- Cooldown phase for the last 20 steps

### Batch Configuration

```yaml
resolution: 1008
max_annotations_per_image: 200
batch_size: varies by GPU count
```

### Training Schedule

The training uses a sophisticated multi-component learning rate schedule with layer-wise decay for the vision backbone.

## 5. Dataset Preparation

SAM3 training supports multiple dataset formats:

### Roboflow 100-VL

**Structure:**
```
dataset_root/
├── 13-lkc01/
│   ├── train/
│   ├── valid/
│   └── test/
├── 2024-frc/
│   ├── train/
│   ├── valid/
│   └── test/
└── ...
```

### ODinW13

**Structure:**
```
dataset_root/
├── AerialMaritimeDrone/
│   └── large/
│       ├── train/
│       ├── valid/
│       └── test/
└── ...
```

### COCO Format

SAM3 also supports **COCO-style annotations**:

```json
{
  "images": [
    {
      "id": 1,
      "file_name": "image_001.png",
      "width": 1024,
      "height": 768
    }
  ],
  "annotations": [
    {
      "id": 1,
      "image_id": 1,
      "category_id": 1,
      "bbox": [x, y, w, h],
      "area": 12345,
      "segmentation": {"counts": "...", "size": [h, w]},
      "iscrowd": 0
    }
  ],
  "categories": [
    {
      "id": 1,
      "name": "concept_name",
      "supercategory": "concept"
    }
  ]
}
```

## 6. Configuration Files

Training configurations are structured in YAML format with four main sections:

### 1. Training Configuration
```yaml
training:
  batch_size: 32
  learning_rate: 8e-5
  optimizer: AdamW
  weight_decay: 0.1
  gradient_clip: 0.1
```

### 2. Model Configuration
```yaml
model:
  checkpoint_path: "path/to/checkpoint"
  enable_segmentation: true
  enable_inst_interactivity: false
```

### 3. Launcher Configuration
```yaml
launcher:
  distributed: true
  num_gpus: 4
  use_cluster: false
```

### 4. Logging Configuration
```yaml
logging:
  output_dir: "./outputs"
  log_frequency: 100
  save_checkpoint_frequency: 1000
```

### Example Configuration Reference

The repository includes example configs like:
- `roboflow_v100_full_ft_100_images.yaml`: Full fine-tuning on 100 images

## 7. Fine-Tuning SAM3 on Custom Datasets

### Approach

SAM3 is designed for **fine-tuning** rather than training from scratch. The recommended approach is transfer learning from the pre-trained checkpoint.

### Steps for Fine-Tuning

#### Step 1: Prepare Your Dataset

Convert your data to COCO format with proper annotations:
- Images in a folder
- Annotations JSON with segmentation masks
- Category labels

#### Step 2: Create Configuration File

```yaml
# custom_finetune.yaml
model:
  checkpoint: "path/to/sam3_checkpoint.pth"
  enable_segmentation: true

dataset:
  root: "path/to/your/data"
  annotation_file: "annotations.json"
  
training:
  batch_size: 16
  learning_rate: 8e-5
  num_epochs: 10
  
scratch:
  enable_segmentation: true
```

#### Step 3: Run Fine-Tuning

```bash
python sam3/train/train.py -c custom_finetune --num-gpus 1
```

### Hardware Requirements

- **Minimum**: 1 GPU (NVIDIA with 16GB+ VRAM recommended)
- **Optimal**: 4-8 GPUs for distributed training
- **Large-scale**: Multiple nodes with SLURM support

## 8. What Has Been Done in This Repository

### Existing Implementation

This repository (`fine_tune_SAM`) already includes:

#### 1. SAM3 Auto-Label Pipeline (`nbs/06_sam3_autolabel_pipeline.ipynb`)

A progressive walkthrough notebook that demonstrates:

**Features:**
- Repository inspection and setup
- Dataset loading (breast cancer medical images)
- SAM3 processor initialization
- Automatic mask generation
- COCO-format export for training

**Key Components:**

1. **`AutoLabeler` Class**: Generates pseudo-labels using SAM3 or fallback methods
2. **COCO Export**: Converts masks to COCO format with RLE encoding
3. **Visualization**: Overlay masks on images for quality checking
4. **Configuration**: Environment-based SAM3 enablement (`RUN_SAM3=1`)

**Workflow:**
```python
# 1. Initialize configuration
cfg = AutoLabelConfig(
    repo_root=Path("/workspace/sam3"),
    output_root=Path("/workspace/data/autolabel"),
    prompt_template="breast tumor",
    enable_sam3=True
)

# 2. Initialize SAM3 processor
processor = init_sam3_processor(cfg)

# 3. Create auto-labeler
labeler = AutoLabeler(cfg, processor)

# 4. Generate labels
outputs = [labeler.label_image(record) for record in sample_records]

# 5. Export to COCO format
coco_dict = build_coco_dict(outputs, dataset_name="breast-tumor")
write_json(coco_dict, DATA_ROOT / "autolabel_annotations.json")
```

**Output Structure:**
```
/workspace/data/autolabel/
├── images/
│   ├── sample_00001.png
│   ├── sample_00002.png
│   └── ...
├── masks/
│   ├── sample_00001_mask.png
│   ├── sample_00002_mask.png
│   └── ...
└── autolabel_annotations.json
```

#### 2. Other Notebooks

- `01_patch_dataset.ipynb`: Patch-based dataset preparation
- `02_data_preparation.ipynb`: General data preparation utilities
- `03_upload_data_hf.ipynb`: HuggingFace dataset upload
- `04_data_prep_from_hf.ipynb`: Loading data from HuggingFace
- `05_inference_with_medSAM.ipynb`: Medical SAM inference

### Repository Structure

```
fine_tune_SAM/
├── nbs/                    # Jupyter notebooks
│   └── 06_sam3_autolabel_pipeline.ipynb
├── fine_tune_SAM/          # Python package
│   ├── core.py
│   ├── data_prep.py
│   ├── hf_dataset.py
│   ├── med_sam_inference.py
│   └── patch_dataset.py
├── README.md
└── setup.py
```

## 9. Next Steps and Recommendations

### To Start Training/Fine-Tuning

1. **Get SAM3 Checkpoint**
   - Request access on HuggingFace: `facebook/sam3`
   - Download the checkpoint

2. **Clone SAM3 Repository**
   ```bash
   cd /workspace
   git clone https://github.com/facebookresearch/sam3.git
   cd sam3
   pip install -e ".[train,dev]"
   ```

3. **Generate Pseudo-Labels**
   - Run the `06_sam3_autolabel_pipeline.ipynb` notebook with `RUN_SAM3=1`
   - This will generate COCO-format annotations

4. **Create Training Configuration**
   - Copy an example config from `sam3/train/configs/`
   - Point `dataset.root` to your data directory
   - Set `scratch.enable_segmentation: true` for pixel-level losses

5. **Run Fine-Tuning**
   ```bash
   python sam3/train/train.py -c your_config --num-gpus 1
   ```

### Quality Improvements

Extend the `AutoLabeler` class with:
- Score thresholds to filter low-confidence predictions
- Minimum area filters to avoid noisy small masks
- Multi-prompt strategies for better coverage
- Active learning loops for iterative improvement

### Reproducibility

- Commit generated JSON + PNG assets to version control
- Or push to HuggingFace Datasets for sharing
- Track hyperparameters and training curves

## 10. References and Resources

### Official Resources

- **GitHub Repository**: [facebookresearch/sam3](https://github.com/facebookresearch/sam3)
- **Meta AI Page**: [ai.meta.com/sam3](https://ai.meta.com/sam3/)
- **HuggingFace Model**: [facebook/sam3](https://huggingface.co/facebook/sam3)
- **ArXiv Paper**: [2511.16719 - SAM 3: Segment Anything with Concepts](https://arxiv.org/abs/2511.16719)

### Training Documentation

- **Training Guide**: `README_TRAIN.md` in the SAM3 repository
- **Example Configs**: `sam3/train/configs/`

### Datasets

- **SA-Co Dataset**: HuggingFace and Roboflow
- **Roboflow 100-VL**: Vision-language detection dataset
- **ODinW13**: Object detection in the wild

### Community Resources

- **Ultralytics Integration**: [docs.ultralytics.com/models/sam-3](https://docs.ultralytics.com/models/sam-3/)
- **Blog Posts**: Multiple tutorials on Roboflow, LearnOpenCV, etc.
- **GitHub Issues**: Active community discussion on the repository

### Related Models

- **SAM 1**: Original Segment Anything Model
- **SAM 2**: Added video segmentation capabilities
- **SAM 3D**: 3D object reconstruction from single images
- **EfficientSAM3**: Distilled lightweight version

## Summary

This notebook documents the complete SAM3 training ecosystem:

**Key Takeaways:**

1. SAM3 is Meta's 848M parameter open-vocabulary segmentation model
2. Training uses AdamW optimizer with component-specific learning rates (8e-5 for transformer)
3. Fine-tuning approach is recommended over training from scratch
4. COCO format annotations are supported
5. The existing auto-label pipeline provides pseudo-label generation
6. Multi-GPU distributed training is supported

**Training Parameters Summary:**
- Optimizer: AdamW
- Learning Rate: 8e-5 (transformer), 2.5e-5 (vision), 5e-6 (language)
- Weight Decay: 0.1
- Gradient Clipping: 0.1
- Resolution: 1008
- Scheduler: InverseSquareRoot with warmup

The repository is ready for fine-tuning once the SAM3 checkpoint is obtained from HuggingFace.